In [1]:
import pandas as pd
import numpy as np


In [2]:
day="7th August"
data=pd.read_csv('aug7.csv')
data.head()

,Timestamp,Name,Roll no,Distance
0,8/7/2026 18:37:55,Dhairya Gera,b26035,12 meters away
1,8/7/2026 18:39:56,Vidya Bharti,B26114,10 meters away
2,8/7/2026 18:39:56,Paarth Vashistha,B26458,12 meters away
3,8/7/2026 18:40:00,Shreyas Santosh Bhegade,B26270,9 meters away
4,8/7/2026 18:40:40,Pankaj,B26145,12 meters away


In [3]:
output=pd.read_csv("complete_attendance_v1.csv")
output.head()

,Roll Number,3rd August,4th August,5th August,6th August,7th August,8th August,9th August
0,B26001,1,1,1,1,0,0,0
1,B26002,1,1,1,1,0,0,0
2,B26003,1,1,1,1,0,0,0
3,B26004,1,1,1,1,0,0,0
4,B26005,1,1,1,1,0,0,0


In [5]:

roll=data['Roll no'].tolist()
for i in range(len(roll)):
    roll[i]=str(roll[i])
    roll[i]=roll[i].strip()
    roll[i]=roll[i].upper()
print(roll)
    
    

['B26035', 'B26114', 'B26458', 'B26270', 'B26145', 'B26608', 'B26618', 'B26271', 'B26411', 'B26583', 'B24497', 'B26347', 'B26515', 'B26029', 'B26519', 'B26298', 'B26120', 'B26382', 'B26257', 'B26356', 'B26419', 'B26487', 'B26622', 'B26358', 'B26070', 'B26315', 'B26340', 'B26590', 'B26455', 'IM26001', 'IM26013', 'B26258', 'B26592', 'B26302', 'B26146', 'B26431', 'B26259', 'B26141', 'B26211', 'B26430', 'B26110', 'B26334', 'B26451', 'B26620', 'B26291', 'B26175', 'B26563', 'B26438', 'B26461', 'B26481', 'B26229', 'IM26037', 'B26387', 'B26135', 'B26130', 'B26625', 'B26268', 'B26553', 'B26565', 'B26052', 'B26042', 'B26278', 'B26460', 'B26099', 'B26041', 'B26015', 'B26629', 'B26140', 'B26063', 'B26293', 'B26196', 'B26586', 'B26544', 'B26442', 'B26495', 'B26574', 'B26183', 'B26111', 'B26423', 'B26241', 'B26113', 'B26593', 'B26106', 'B26174', 'B26568', 'B26504', 'B26266', 'B26122', 'B26209', 'B26008', 'IM26004', 'B26236', 'B26137', 'B26441', 'B26465', 'B26310', 'B26322', 'B26089', 'B26435', 'B264

In [6]:
def generate_attendance():
    for i in roll:
        output.loc[output['Roll Number']==i,day]=1
    print(output.head(100))
    output.to_csv("complete_attendance_v1.csv",index=False)   
        

In [7]:
generate_attendance()

   Roll Number 3rd August  4th August  5th August  6th August  7th August  \
0       B26001          1           1           1           1           1   
1       B26002          1           1           1           1           1   
2       B26003          1           1           1           1           1   
3       B26004          1           1           1           1           1   
4       B26005          1           1           1           1           1   
..         ...        ...         ...         ...         ...         ...   
95      B26096          1           1           1           1           1   
96      B26097          1           1           1           1           1   
97      B26098          0           0           0           0           0   
98      B26099          1           1           1           1           1   
99      B26100          1           1           1           1           1   

    8th August  9th August  
0            0           0  
1            0   

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

def analyze_and_plot_attendance(file_path, day='3rd August'):
    # 1. Load Data
    df = pd.read_csv(file_path)

    # 2. Categorize Program based on Roll Number prefix
    def get_program(roll):
        roll_str = str(roll).strip()
        if roll_str.startswith('B26'):
            return 'BTech'
        elif roll_str.startswith('IM26'):
            return 'IMBA'
        else:
            return 'Other'

    df['Program'] = df['Roll Number'].apply(get_program)
    date_cols = [c for c in df.columns if c not in ['Roll Number', 'Program']]

    # Check if specified day exists in the dataset
    if day not in date_cols:
        print(f"Error: '{day}' was not found in the dataset dates.")
        print(f"Available dates: {date_cols}")
        return

    # 3. Reshape Data (Long format)
    df_long = df.melt(
        id_vars=['Roll Number', 'Program'], 
        value_vars=date_cols, 
        var_name='Date', 
        value_name='Status'
    )
    
    # Normalize status labels
    df_long['Status'] = df_long['Status'].astype(str).str.strip().str.capitalize()
    status_map = {'1': 'Present', '0': 'Absent', 'Proxy': 'Proxy'}
    df_long['Status'] = df_long['Status'].map(status_map).fillna(df_long['Status'])

    # Filter data for the requested day
    selected_day_df = df_long[df_long['Date'] == day]

    # --- PRINT SUMMARY STATISTICS ---
    print("=" * 60)
    print(f" SUMMARY STATISTICS FOR {day.upper()}")
    print("=" * 60)
    
    print("\n1. TOTAL STUDENT COUNT BY PROGRAM:")
    print(df['Program'].value_counts().to_string())

    print(f"\n2. ATTENDANCE SUMMARY FOR {day}:")
    day_summary = selected_day_df.groupby(['Program', 'Status']).size().unstack(fill_value=0)
    print(day_summary.to_string())

    print(f"\n3. PROXY INCIDENTS DETECTED ON {day}:")
    proxies = selected_day_df[selected_day_df['Status'] == 'Proxy']
    if not proxies.empty:
        print(proxies[['Roll Number', 'Program', 'Date']].to_string(index=False))
    else:
        print(f"No proxy attendance found on {day}.")
    print("=" * 60)

    # File naming helper (converts "3rd August" -> "3rd_august")
    safe_day_str = day.lower().replace(' ', '_')

    # --- GENERATE & SAVE GRAPHS ---

    # Graph 1: Program Distribution (Pie Chart)
    plt.figure(figsize=(6, 6))
    program_counts = df['Program'].value_counts()
    plt.pie(
        program_counts, 
        labels=program_counts.index, 
        autopct='%1.1f%%', 
        colors=['#4C72B0', '#DD8452'], 
        startangle=140, 
        explode=(0.05, 0)
    )
    plt.title('Student Distribution by Program', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('program_distribution.png', dpi=300)
    plt.close()
    print(" Saved: program_distribution.png")

    # Graph 2: Attendance Comparison on Specified Day (Bar Chart)
    plt.figure(figsize=(8, 5))
    ax = sns.countplot(
        data=selected_day_df, 
        x='Program', 
        hue='Status', 
        palette={'Present': '#55A868', 'Absent': '#C44E52', 'Proxy': '#8172B3'}
    )
    plt.title(f'Attendance Status Comparison ({day})', fontsize=14, fontweight='bold')
    plt.xlabel('Program', fontsize=12)
    plt.ylabel('Number of Students', fontsize=12)
    
    # Add data values on top of bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')
    
    plt.tight_layout()
    day_bar_filename = f'{safe_day_str}_attendance.png'
    plt.savefig(day_bar_filename, dpi=300)
    plt.close()
    print(f" Saved: {day_bar_filename}")

    # Graph 3: Date-wise Attendance Trend (Line Chart across all days)
    plt.figure(figsize=(10, 5))
    trend_df = df_long.groupby(['Date', 'Status']).size().reset_index(name='Count')
    sns.lineplot(
        data=trend_df, 
        x='Date', 
        y='Count', 
        hue='Status', 
        marker='o', 
        linewidth=2.5, 
        palette={'Present': '#55A868', 'Absent': '#C44E52', 'Proxy': '#8172B3'}
    )
    plt.title('Overall Attendance Trend Across All Dates', fontsize=14, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Total Count', fontsize=12)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig('attendance_trend.png', dpi=300)
    plt.close()
    print(" Saved: attendance_trend.png")

# Run analysis for a specific day
if __name__ == "__main__":
    # Change 'day' variable to any date present in your dataset 
    # Available options: '3rd August', '4th August', '5th August', '6th August', '7th August', '8th August', '9th August'
    target_day = "7th August"
    
    analyze_and_plot_attendance('complete_attendance_v1.csv', day=target_day)

/home/remandey/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


 SUMMARY STATISTICS FOR 7TH AUGUST

1. TOTAL STUDENT COUNT BY PROGRAM:
Program
BTech    640
IMBA      80

2. ATTENDANCE SUMMARY FOR 7th August:
Status   Absent  Present
Program                 
BTech        80      560
IMBA         19       61

3. PROXY INCIDENTS DETECTED ON 7th August:
No proxy attendance found on 7th August.
 Saved: program_distribution.png
 Saved: 7th_august_attendance.png
 Saved: attendance_trend.png
